In [1]:
import csv
f = open("tanker_scheduling.csv")
csvfile = csv.DictReader(f, delimiter='\t')

headers = csvfile.fieldnames

table = []
for row in csvfile:
    table.append(row)
    
f.close()

In [2]:
print(headers)

['Terminal', 'Travel Time', 'Order 1', 'Order 2', 'Order 3', 'Order 4', 'Order 5', 'Order 6']


In [3]:
# read the parameters as floats
TravelTime = {}
Orders = {}
for row in table:
    TravelTime[row['Terminal']] = float(row['Travel Time'])
    # to read the orders, loop over 1 through 6 and only add non-empty order dates
    tmpOrders = set()
    for i in range(1,7):
        if row["Order "+str(i)] != "":
            tmpOrders.add(float(row["Order "+str(i)]))
    # add the orders to this terminal
    Orders[row['Terminal']] = tmpOrders

# be cautious: these dates are not ordered.  Check if this is needed.

In [4]:
print(TravelTime)
print(Orders)

{'Barcelona, Spain': 22.0, 'Gujarat, India': 5.0, 'Gwangyang, South Korea': 31.0, 'Milford Haven, UK': 28.0, 'Tokyo, Japan': 33.0, 'Zeebrugge, Belgium': 29.0}
{'Barcelona, Spain': {57.0, 99.0, 117.0, 30.0}, 'Gujarat, India': {67.0, 100.0, 37.0, 75.0, 25.0, 60.0}, 'Gwangyang, South Korea': {106.0, 45.0, 115.0, 83.0, 55.0}, 'Milford Haven, UK': {98.0, 72.0, 46.0, 111.0, 90.0, 59.0}, 'Tokyo, Japan': {67.0, 44.0, 109.0, 79.0, 87.0}, 'Zeebrugge, Belgium': {32.0, 89.0, 114.0, 63.0}}


In [5]:
# for clarity, define a set for the terminals (borrow key from TravelTime)
Terminals = { t for t in TravelTime }

# define the set of nodes: one for each terminal/order pair
# - we can enumerate these as numbers {1,2,...}
# - but then we need to define Terminal(i) and OrderDate(i) for each i in the set of nodes
Terminal = {}
OrderDate = {}
for t in Terminals:
    for o in Orders[t]:
        newNode = len(Terminal)+1
        Terminal[newNode] = t
        OrderDate[newNode] = o

Nodes = { i for i in Terminal }
        
# define arcs: (i,j) exists if delivery(i) + distance(i, Qatar) + distance(Qatar, j)  ≤ delivery(j)
Arcs = { (i,j) for i in Nodes for j in Nodes 
        if OrderDate[i] + TravelTime[Terminal[i]] + TravelTime[Terminal[j]] <= OrderDate[j] }

In [6]:
# complete the network by adding 'source' and 'sink' nodes and associated arcs.
Nodes.add('source')
Nodes.add('sink')

In [7]:
# connect each node to the source and to the sink
for i in Nodes:
    if i != 'source' and i != 'sink':
        Arcs.add(('source',i))
        Arcs.add((i,'sink'))

In [8]:
# add one more arc from sink to source (representing the total flow through the network)
Arcs.add(('sink','source'))

In [9]:
# build the optimization model
from docplex.mp.model import Model
mdl = Model()

In [10]:
# variables

# Note: For Python 3.7 we can use the set Arcs directly as dictionary 
#       but for Python 3.8 or later the dictionary needs to be ordered.
#       For this reason, we convert the set into a list 'ArcsDict'.
ArcsDict = list(Arcs)
flow = mdl.continuous_var_dict(ArcsDict, lb=0, name='flow')

In [11]:
# objective
mdl.minimize(flow[('sink','source')])

In [12]:
# constraints: flow conservation
for j in Nodes:
    inflow = mdl.sum(flow[i,j] for i in Nodes if (i,j) in Arcs)
    outflow = mdl.sum(flow[j,i] for i in Nodes if (j,i) in Arcs)
    mdl.add_constraint( inflow - outflow == 0 )
    
    # constraints: make sure each delivery order (i.e., node) is visited exactly once
    # (re-use the inflow for this purpose)
    if j != 'source' and j != 'sink':
        mdl.add_constraint(inflow == 1)

In [13]:
# solve
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0,status='optimal')

In [14]:
mdl.print_information()

Model: docplex_model1
 - number of variables: 185
   - binary=0, integer=0, continuous=185
 - number of constraints: 62
   - linear=62
 - parameters: defaults
 - problem type is: LP


In [15]:
mdl.print_solution()

flow_sink_source: 17.000
  flow_30_26=1.000
  flow_source_17=1.000
  flow_21_9=1.000
  flow_9_sink=1.000
  flow_source_8=1.000
  flow_source_7=1.000
  flow_source_18=1.000
  flow_19_sink=1.000
  flow_23_10=1.000
  flow_source_13=1.000
  flow_4_sink=1.000
  flow_3_12=1.000
  flow_14_sink=1.000
  flow_10_sink=1.000
  flow_26_24=1.000
  flow_7_sink=1.000
  flow_8_28=1.000
  flow_source_3=1.000
  flow_source_30=1.000
  flow_1_sink=1.000
  flow_16_sink=1.000
  flow_source_25=1.000
  flow_source_20=1.000
  flow_2_sink=1.000
  flow_source_15=1.000
  flow_25_23=1.000
  flow_5_sink=1.000
  flow_24_19=1.000
  flow_15_sink=1.000
  flow_27_4=1.000
  flow_source_5=1.000
  flow_source_16=1.000
  flow_17_21=1.000
  flow_11_sink=1.000
  flow_source_11=1.000
  flow_source_6=1.000
  flow_20_29=1.000
  flow_source_27=1.000
  flow_6_22=1.000
  flow_12_sink=1.000
  flow_source_1=1.000
  flow_29_sink=1.000
  flow_sink_source=17.000
  flow_22_sink=1.000
  flow_18_sink=1.000
  flow_13_14=1.000
  flow_28_sink=

In [16]:
# post processing: print the schedule for each of the vessels in the solution

# copy the arcs in the flow solution
FlowArcs = { a for a in Arcs 
            if flow[a].solution_value >= 0.99 and flow[a].solution_value <= 1.01 }
   
# create a usable format to track the `schedule' for each vessel
NextNode = {}
for (i,j) in FlowArcs:
    NextNode[i] = j

In [17]:
# define the set of initial visits
FirstVisits = { j for (i,j) in FlowArcs if i=='source' }

# follow the next node for each node in FirstVisit until the sink is reached
for i in FirstVisits:
    print("\nStart new schedule:")
    print(Terminal[i]+" ("+str(OrderDate[i])+")")
    j = NextNode[i]
    while j != 'sink':
        print(Terminal[j]+" ("+str(OrderDate[j])+")")
        j = NextNode[j]


Start new schedule:
Milford Haven, UK (98.0)

Start new schedule:
Milford Haven, UK (72.0)

Start new schedule:
Milford Haven, UK (46.0)
Gwangyang, South Korea (106.0)

Start new schedule:
Milford Haven, UK (90.0)

Start new schedule:
Milford Haven, UK (59.0)
Gujarat, India (100.0)

Start new schedule:
Tokyo, Japan (67.0)

Start new schedule:
Tokyo, Japan (44.0)
Barcelona, Spain (99.0)

Start new schedule:
Tokyo, Japan (87.0)

Start new schedule:
Gwangyang, South Korea (45.0)
Gwangyang, South Korea (115.0)

Start new schedule:
Gwangyang, South Korea (83.0)

Start new schedule:
Gwangyang, South Korea (55.0)

Start new schedule:
Zeebrugge, Belgium (32.0)
Gujarat, India (67.0)
Tokyo, Japan (109.0)

Start new schedule:
Zeebrugge, Belgium (89.0)

Start new schedule:
Zeebrugge, Belgium (63.0)
Barcelona, Spain (117.0)

Start new schedule:
Gujarat, India (25.0)
Gujarat, India (37.0)
Tokyo, Japan (79.0)

Start new schedule:
Barcelona, Spain (57.0)
Milford Haven, UK (111.0)

Start new schedule: